[Énoncés : Série 2 — Matrices et simulation](../serie2_matrices_simulation.ipynb)

# Corrigé — Série 2 — Matrices et simulation

Pour chaque exercice : l'idée, la complexité, puis une solution commentée avec ses tests. Il existe souvent plusieurs bonnes solutions : si la tienne passe tous les tests et le test de vitesse, elle est juste. Compare surtout l'**idée**.

Exécute d'abord la cellule d'outils.

In [ ]:
# Outils de ce notebook : exécute cette cellule en premier.
# - verifier(fonction, cas) lance ta fonction sur chaque cas et affiche ✅ (bon), ❌ (mauvais) ou 💥 (plantage).
# - chrono(fonction, ...) mesure le temps sur une grande entrée (tests de vitesse facultatifs).
# Tu n'as pas besoin de comprendre ce code pour t'en servir.

import copy
import time

def _court(texte, limite=100):
    return texte if len(texte) <= limite else texte[:limite - 3] + "..."

def verifier(fonction, cas, ne_pas_modifier=False):
    reussis = 0
    for arguments, attendu in cas:
        texte_args = _court(", ".join(repr(a) for a in arguments))
        copie = copy.deepcopy(arguments)
        try:
            obtenu = fonction(*copie)
        except Exception as erreur:
            print(f"💥 {fonction.__name__}({texte_args}) a planté : {type(erreur).__name__}: {erreur}")
            continue
        if obtenu != attendu:
            print(f"❌ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
        elif ne_pas_modifier and copie != arguments:
            print(f"⚠️ {fonction.__name__}({texte_args}) -> bon résultat, mais ta fonction a modifié son entrée (interdit ici)")
        else:
            reussis += 1
            print(f"✅ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}")
    print(f"\n{reussis}/{len(cas)} tests réussis")

def chrono(fonction, *arguments, attendu=None, limite=1.0):
    debut = time.perf_counter()
    try:
        obtenu = fonction(*arguments)
    except Exception as erreur:
        print(f"💥 {fonction.__name__} a planté : {type(erreur).__name__}: {erreur}")
        return
    duree = time.perf_counter() - debut
    if obtenu is None:
        print("⏳ Ta fonction ne renvoie rien pour l'instant : écris-la d'abord.")
    elif attendu is not None and obtenu != attendu:
        print(f"❌ Résultat faux sur la grande entrée : {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
    elif duree <= limite:
        print(f"✅ {duree:.3f} s : assez rapide")
    else:
        print(f"⚠️ {duree:.1f} s : trop lent (limite : {limite} s). Ta solution n'a probablement pas la complexité attendue.")

## Exercice 7 — Somme de la bordure

**Idée** : parcourir chaque case **une seule fois** et tester si elle est sur un bord. Additionner « première ligne + dernière ligne + première colonne + dernière colonne » compterait les coins deux fois, et la ligne entière deux fois quand il n'y a qu'une ligne.

**Complexité** : O(lignes × colonnes).

In [ ]:
def somme_bordure(grille: list[list[int]]) -> int:
    lignes, colonnes = len(grille), len(grille[0])
    total = 0
    for i in range(lignes):
        for j in range(colonnes):
            # une case est sur le bord si sa ligne OU sa colonne est une extrémité
            if i == 0 or i == lignes - 1 or j == 0 or j == colonnes - 1:
                total += grille[i][j]
    return total


verifier(somme_bordure, [
    (([[1, 2, 3], [4, 5, 6], [7, 8, 9]],), 40),
    (([[5]],), 5),   # une seule case
    (([[1, 2, 3]],), 6),   # une seule ligne
    (([[1], [2], [3]],), 6),   # une seule colonne
    (([[1, 2], [3, 4]],), 10),
    (([[1, 1, 1, 1], [1, 9, 9, 1], [1, 9, 9, 1], [1, 1, 1, 1]],), 12),
    (([[2, 3], [4, 5], [6, 7]],), 27),
])

## Exercice 8 — La gravité dans les colonnes

**Idée** : traiter les colonnes une par une. Pour chaque colonne, récupérer les blocs dans l'ordre (de haut en bas), puis les reposer **en bas** de la colonne, dans le même ordre. On écrit dans une nouvelle grille : l'originale n'est jamais modifiée.

**Complexité** : O(lignes × colonnes).

In [ ]:
def gravite_colonnes(grille: list[list[int]]) -> list[list[int]]:
    lignes, colonnes = len(grille), len(grille[0])
    resultat = [[0] * colonnes for _ in range(lignes)]         # nouvelle grille, vide
    for j in range(colonnes):
        blocs = [grille[i][j] for i in range(lignes) if grille[i][j] != 0]   # de haut en bas
        depart = lignes - len(blocs)                           # on les pose tout en bas
        for k, bloc in enumerate(blocs):
            resultat[depart + k][j] = bloc
    return resultat


verifier(gravite_colonnes, [
    (([[1, 0, 2], [0, 3, 0], [4, 0, 5]],), [[0, 0, 0], [1, 0, 2], [4, 3, 5]]),
    (([[0]],), [[0]]),
    (([[7], [0], [0]],), [[0], [0], [7]]),
    (([[1, 2], [3, 4]],), [[1, 2], [3, 4]]),   # colonnes pleines : rien ne bouge
    (([[5, 0], [0, 0], [6, 0], [0, 8]],), [[0, 0], [0, 0], [5, 0], [6, 8]]),
], ne_pas_modifier=True)

## Exercice 9 — Robot sur une grille

**Idée** : un dict « commande → décalage `(di, dj)` », comme les directions du thème 2. On calcule la case visée, et on ne bouge que si elle est dans la grille.

**Complexité** : O(nombre de commandes).

In [ ]:
def robot_grille(lignes: int, colonnes: int, commandes: str) -> tuple[int, int]:
    deplacements = {"N": (-1, 0), "S": (1, 0), "E": (0, 1), "O": (0, -1)}
    i, j = 0, 0
    for c in commandes:
        di, dj = deplacements[c]
        ni, nj = i + di, j + dj
        if 0 <= ni < lignes and 0 <= nj < colonnes:   # sinon : déplacement ignoré
            i, j = ni, nj
    return (i, j)


verifier(robot_grille, [
    ((3, 4, 'EESSON'), (1, 1)),
    ((3, 4, ''), (0, 0)),   # aucune commande
    ((3, 4, 'NNOO'), (0, 0)),   # tout est ignoré
    ((2, 2, 'SSSEEE'), (1, 1)),
    ((1, 5, 'EEEEEEEO'), (0, 3)),
    ((4, 4, 'SSSSEEN'), (2, 2)),
])

## Exercice 10 — Le carré 2 × 2 le plus chargé

**Idée** : essayer chaque coin haut-gauche possible. En parcourant ligne par ligne puis de gauche à droite, le premier carré trouvé est le plus haut puis le plus à gauche : avec une comparaison **stricte** (`>`), on le garde en cas d'égalité. Piège : initialiser le meilleur à `0` échoue quand toutes les sommes sont négatives.

**Complexité** : O(lignes × colonnes).

In [ ]:
def meilleur_carre_2x2(grille: list[list[int]]) -> list[int]:
    meilleur = None
    for i in range(len(grille) - 1):
        for j in range(len(grille[0]) - 1):
            somme = grille[i][j] + grille[i][j + 1] + grille[i + 1][j] + grille[i + 1][j + 1]
            if meilleur is None or somme > meilleur[2]:   # > strict : à égalité, on garde le premier trouvé
                meilleur = [i, j, somme]
    return meilleur


verifier(meilleur_carre_2x2, [
    (([[1, 2, 3], [4, 5, 6], [7, 8, 9]],), [1, 1, 28]),
    (([[1, 1], [1, 1]],), [0, 0, 4]),
    (([[0, 0, 0], [0, 0, 0]],), [0, 0, 0]),   # égalité : le plus haut, puis le plus à gauche
    (([[9, 9, 0], [9, 9, 0], [0, 0, 0]],), [0, 0, 36]),
    (([[-1, -2], [-3, -4]],), [0, 0, -10]),   # sommes négatives
    (([[0, 0, 0], [0, 5, 5], [0, 5, 5]],), [1, 1, 20]),
])

## Exercice 11 — Une minute de diffusion

**Idée** : c'est exactement `un_tour` du thème 2. On **lit** l'ancienne grille et on **écrit** dans une copie ; sinon, un `0` devenu `1` contaminerait sa voisine dans la même minute. La copie se fait ligne par ligne (`ligne[:]`) : `grille[:]` ou `grille.copy()` ne copieraient pas les lignes.

**Complexité** : O(lignes × colonnes).

In [ ]:
def diffusion_une_etape(grille: list[list[int]]) -> list[list[int]]:
    lignes, colonnes = len(grille), len(grille[0])
    nouvelle = [ligne[:] for ligne in grille]          # on LIT grille, on ÉCRIT nouvelle
    for i in range(lignes):
        for j in range(colonnes):
            if grille[i][j] == 0:
                for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    ni, nj = i + di, j + dj
                    if 0 <= ni < lignes and 0 <= nj < colonnes and grille[ni][nj] == 1:
                        nouvelle[i][j] = 1
                        break
    return nouvelle


verifier(diffusion_une_etape, [
    (([[1, 0, 0, 0]],), [[1, 1, 0, 0]]),
    (([[0, 0], [0, 0]],), [[0, 0], [0, 0]]),
    (([[0, 0, 0], [0, 1, 0], [0, 0, 0]],), [[0, 1, 0], [1, 1, 1], [0, 1, 0]]),
    (([[1]],), [[1]]),
    (([[1, 0], [0, 0]],), [[1, 1], [1, 0]]),   # pas de diagonale
    (([[0], [0], [1]],), [[0], [1], [1]]),
], ne_pas_modifier=True)

## Exercice 12 — Périmètre des terres

**Idée** : pour chaque case de terre, regarder ses 4 côtés. Un côté compte si la case voisine est hors de la grille ou de l'eau. Variante : `4 × (nombre de terres) − 2 × (nombre de paires de terres voisines)`.

**Complexité** : O(lignes × colonnes).

In [ ]:
def perimetre_terres(grille: list[list[int]]) -> int:
    lignes, colonnes = len(grille), len(grille[0])
    perimetre = 0
    for i in range(lignes):
        for j in range(colonnes):
            if grille[i][j] == 1:
                for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    ni, nj = i + di, j + dj
                    # ce côté compte s'il donne sur l'extérieur ou sur de l'eau
                    if not (0 <= ni < lignes and 0 <= nj < colonnes) or grille[ni][nj] == 0:
                        perimetre += 1
    return perimetre


verifier(perimetre_terres, [
    (([[1, 1], [1, 0]],), 8),
    (([[0]],), 0),
    (([[1]],), 4),
    (([[1, 0, 1]],), 8),   # deux îles
    (([[1, 1], [1, 1]],), 8),
    (([[0, 1, 0, 0], [1, 1, 1, 0], [0, 1, 0, 0], [1, 1, 0, 0]],), 16),
])

---

[Énoncés de la série](../serie2_matrices_simulation.ipynb)